# Lab 10 — Introduction to Machine Translation
**Completed tasks and executed outputs**

This notebook answers Task 1 and implements Task 2 with a small, self-contained English–French e-commerce dataset. The Kaggle dataset linked in the lab was not supplied, so the example below is a demonstration, not a model trained on the full Kaggle data.

## Task 1 — Understanding the model layers

**1. Embedding:** Converts each word ID (an integer) into a dense vector of learned numbers. Similar words can learn similar representations.

**2. Encoder LSTM:** Reads the English sentence in order and builds a representation of its meaning and context.

**3. Decoder LSTM:** Uses the encoded representation to produce a sequence of French word predictions.

**4. Softmax:** Turns the final scores into probabilities for all possible French words, so the model can choose a word at each step.

## Task 2 — English-to-French translation for e-commerce

**Use case:** Translate product descriptions and customer-facing phrases for French-speaking shoppers.

The lab provides a Kaggle dataset link but does not include the CSV. The code uses a small sample dataset so it can run without external downloads. A production system would require a much larger parallel corpus and proper evaluation.

In [1]:
import random
import numpy as np
import torch
from torch import nn

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(1)
print("PyTorch version:", torch.__version__)

PyTorch version: 2.10.0+cpu


In [2]:
pairs = [
    ("hello", "bonjour"),
    ("thank you", "merci"),
    ("welcome", "bienvenue"),
    ("free shipping", "livraison gratuite"),
    ("fast delivery", "livraison rapide"),
    ("order now", "commandez maintenant"),
    ("new product", "nouveau produit"),
    ("great quality", "excellente qualité"),
    ("good price", "bon prix"),
    ("customer service", "service client"),
    ("out of stock", "rupture de stock"),
    ("add to cart", "ajouter au panier"),
    ("special offer", "offre spéciale"),
    ("secure payment", "paiement sécurisé"),
    ("product review", "avis produit"),
    ("buy now", "achetez maintenant"),
]
print("Number of example translation pairs:", len(pairs))
for en, fr in pairs[:5]:
    print(f"{en} -> {fr}")

Number of example translation pairs: 16
hello -> bonjour
thank you -> merci
welcome -> bienvenue
free shipping -> livraison gratuite
fast delivery -> livraison rapide


In [3]:
def make_vocab(sentences):
    vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
    for sentence in sentences:
        for word in sentence.lower().split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

eng_vocab = make_vocab([en for en, fr in pairs])
fr_vocab = make_vocab([fr for en, fr in pairs])
fr_reverse = {index: word for word, index in fr_vocab.items()}

def encode(sentence, vocab, max_len):
    ids = [vocab.get(word, vocab["<unk>"]) for word in sentence.lower().split()]
    ids = ids[:max_len-1] + [vocab["<eos>"]]
    return ids + [vocab["<pad>"]] * (max_len - len(ids))

source_len = max(len(en.split()) for en, _ in pairs) + 1
target_len = max(len(fr.split()) for _, fr in pairs) + 1
X = torch.tensor([encode(en, eng_vocab, source_len) for en, fr in pairs])
Y = torch.tensor([encode(fr, fr_vocab, target_len) for en, fr in pairs])
print("English input shape:", tuple(X.shape))
print("French target shape:", tuple(Y.shape))

English input shape: (16, 4)
French target shape: (16, 4)


In [4]:
class TranslationModel(nn.Module):
    def __init__(self, source_size, target_size, hidden_size=48):
        super().__init__()
        self.embedding = nn.Embedding(source_size, hidden_size, padding_idx=0)
        self.encoder = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, target_size)

    def forward(self, source, decoder_input):
        embedded_source = self.embedding(source)
        _, (hidden, cell) = self.encoder(embedded_source)
        decoder_embeddings = self.target_embedding(decoder_input)
        decoded, _ = self.decoder(decoder_embeddings, (hidden, cell))
        return self.output(decoded)

    def setup_target_embedding(self, target_size, hidden_size=48):
        self.target_embedding = nn.Embedding(target_size, hidden_size, padding_idx=0)

model = TranslationModel(len(eng_vocab), len(fr_vocab))
model.setup_target_embedding(len(fr_vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
criterion = nn.CrossEntropyLoss(ignore_index=0)
print("Model created successfully")

Model created successfully


In [5]:
decoder_inputs = torch.cat([torch.full((len(pairs), 1), fr_vocab["<sos>"]), Y[:, :-1]], dim=1)

for epoch in range(1, 301):
    model.train()
    optimizer.zero_grad()
    logits = model(X, decoder_inputs)
    loss = criterion(logits.reshape(-1, len(fr_vocab)), Y.reshape(-1))
    loss.backward()
    optimizer.step()
    if epoch in (1, 50, 100, 200, 300):
        print(f"Epoch {epoch:3d} | loss: {loss.item():.4f}")

Epoch   1 | loss: 3.4903
Epoch  50 | loss: 0.0009


Epoch 100 | loss: 0.0004


Epoch 200 | loss: 0.0002


Epoch 300 | loss: 0.0002


In [6]:
def translate(sentence):
    model.eval()
    source = torch.tensor([encode(sentence, eng_vocab, source_len)])
    generated = [fr_vocab["<sos>"]]
    with torch.no_grad():
        for _ in range(target_len):
            decoder = torch.tensor([generated])
            logits = model(source, decoder)
            next_id = int(logits[0, -1].argmax())
            if next_id == fr_vocab["<eos>"]:
                break
            generated.append(next_id)
    return " ".join(fr_reverse[i] for i in generated[1:] if i not in (0, 1, 2))

for sentence in ["free shipping", "fast delivery", "add to cart", "secure payment", "special offer"]:
    print(f"English: {sentence:18s} | French: {translate(sentence)}")

English: free shipping      | French: livraison gratuite
English: fast delivery      | French: livraison rapide
English: add to cart        | French: ajouter au panier
English: secure payment     | French: paiement sécurisé
English: special offer      | French: offre spéciale


In [7]:
correct = sum(translate(en) == fr for en, fr in pairs)
print(f"Training-set exact match: {correct}/{len(pairs)} ({correct / len(pairs):.1%})")
print("Note: This is training accuracy on a tiny sample, NOT a valid estimate of real-world translation quality.")

Training-set exact match: 16/16 (100.0%)
Note: This is training accuracy on a tiny sample, NOT a valid estimate of real-world translation quality.


### Conclusion

The demonstration shows tokenization, an encoder–decoder LSTM, training, and inference for English–French e-commerce phrases. The training-set score measures memorization of the supplied examples, not generalization. For a real e-commerce deployment, use the full linked Kaggle parallel dataset, split it into train/validation/test sets, and evaluate translations on unseen text.